# EIA 191: Annual (RP7) vs Monthly (RP8) Data Comparison

This notebook investigates Github issue #4755

**Goal:** Compare the EIA NGQS annual (RP7) and monthly (RP8) EIA-191 underground natural gas storage datasets to determine whether and how to integrate the annual data into PUDL.

## Background

EIA Form 191 (*Monthly Underground Natural Gas Storage Report*) is collected monthly from all operators of underground natural gas storage fields in the US. The raw data is publicly available through EIA's Natural Gas Information Annual/Monthly Respondent Query System (NGQS) in two forms:

| Report Code | Description | Years Available |
|-------------|-------------|----------------|
| **RP7** | 191 Field Level Storage Data **(Annual)** | 2005–2024 (2023 missing) |
| **RP8** | 191 Field Level Storage Data **(Monthly)** | 2014–2025 |

**PUDL currently ingests only the monthly (RP8) data**, archived at Zenodo [10.5281/zenodo.18115099](https://zenodo.org/records/18115099). The annual (RP7) data is explicitly skipped in the [pudl-archiver](https://github.com/catalyst-cooperative/pudl-archiver).

This notebook:
1. Loads PUDL's monthly (RP8) data from the Zenodo archive
2. Downloads the annual (RP7) data directly from the EIA NGQS API
3. Compares respondents across the two datasets
4. Compares values across the two datasets
5. Quantifies the historical coverage added by the annual data (2005–2013)

## Key Questions
- Are there storage fields/respondents in the annual data **not present** in the monthly data?
- Does the annual data extend meaningful coverage back to 2005 (before monthly data begins in 2014)?
- Is the annual data a snapshot (e.g. December), an average, or an independent survey?
- Should PUDL archive and ingest the annual data?

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import io
import logging
import zipfile
from pathlib import Path

import pandas as pd
import requests

import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

pd.options.display.max_columns = None
pd.options.display.max_rows = 50

## 2. Load Monthly Data (RP8) — as PUDL ingests it

We download directly from the Zenodo archive that PUDL uses, so the data
matches exactly what the pipeline ingests. Each year is a ZIP file containing a
CSV with columns matching PUDL's `column_maps/data.csv`.

**Zenodo record:** [10.5281/zenodo.18115099](https://zenodo.org/records/18115099)  
**PUDL column map:** `src/pudl/package_data/eia191/column_maps/data.csv`

In [ ]:

ZENODO_RECORD_ID = "18115099"
ZENODO_BASE = f"https://zenodo.org/records/{ZENODO_RECORD_ID}/files"
NGQS_BASE = "https://www.eia.gov/naturalgas/ngqs/data/report"

# Column name mapping for RP7 JSON response (single-letter keys → readable names)
RP7_COL_MAP = {
    "a": "year",
    "b": "report_state",
    "c": "gas_field_code",
    "d": "reservoir_code",
    "e": "company_name",
    "f": "field_name",
    "g": "reservoir_name",
    "h": "field_type",
    "i": "county_name",
    "j": "region",
    "k": "status",
    "l": "base_gas",
    "m": "working_gas_capacity_mcf",
    "n": "total_field_capacity_mcf",
    "o": "maximum_daily_delivery_mcf",
}

# Discover available years from the EIA NGQS metadata API
ngqs_reports = requests.get(NGQS_BASE, timeout=30).json()
ngqs_by_code = {r["code"]: r for r in ngqs_reports}

rp7_meta = ngqs_by_code["RP7"]
rp8_meta = ngqs_by_code["RP8"]

RP7_YEARS = sorted(y["ayear"] for y in rp7_meta["availableYears"])
RP8_YEARS = sorted(y["ayear"] for y in rp8_meta["availableYears"])

print(f"RP7 (Annual) — {rp7_meta['description']}")
print(f"  Last updated: {rp7_meta['last_updated']}")
print(f"  Available years: {RP7_YEARS}")

print(f"\nRP8 (Monthly) — {rp8_meta['description']}")
print(f"  Last updated: {rp8_meta['last_updated']}")
print(f"  Available years: {RP8_YEARS}")


In [ ]:

# Rename RP8 columns that use "(mcf)" suffix to match RP7's "_mcf" convention
RP8_COL_NORMALIZE = {
    "working_gas_capacity_(mcf)": "working_gas_capacity_mcf",
    "total_field_capacity_(mcf)": "total_field_capacity_mcf",
    "maximum_daily_delivery_(mcf)": "maximum_daily_delivery_mcf",
}


def load_rp8_year_from_zenodo(year: int) -> pd.DataFrame:
    """Download and parse one year of RP8 monthly data from the PUDL Zenodo archive."""
    url = f"{ZENODO_BASE}/eia191-{year}.zip"
    logger.info(f"Downloading RP8 {year} from {url}")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        csv_names = [n for n in zf.namelist() if n.endswith(".csv")]
        if not csv_names:
            raise ValueError(f"No CSV found in eia191-{year}.zip: {zf.namelist()}")
        logger.info(f"  Files in archive: {zf.namelist()}")
        with zf.open(csv_names[0]) as f:
            df = pd.read_csv(f)

    df["report_year"] = year
    return df.rename(columns=RP8_COL_NORMALIZE)


# Load a sample year to inspect structure
rp8_sample = load_rp8_year_from_zenodo(2022)
print(f"Shape: {rp8_sample.shape}")
print(f"Columns: {rp8_sample.columns.tolist()}")
rp8_sample.head()


In [ ]:
%%time
# Load all available RP8 years
rp8_all = pd.concat(
    [load_rp8_year_from_zenodo(yr) for yr in RP8_YEARS],
    ignore_index=True,
)
print(f"RP8 total shape: {rp8_all.shape}")
rp8_all.dtypes

## 3. Download Annual Data (RP7) — directly from EIA NGQS

The annual EIA 191 data is NOT currently archived by PUDL. We download it
directly from the EIA NGQS API. The endpoint returns JSON with single-letter
column names that we decode.

**API pattern:** `https://www.eia.gov/naturalgas/ngqs/data/report/RP7/data/{year}/{year}/ICA/Name`

In [ ]:
def load_rp7_year(year: int) -> pd.DataFrame:
    """Download and parse one year of RP7 annual data from EIA NGQS API."""
    url = f"{NGQS_BASE}/RP7/data/{year}/{year}/ICA/Name"
    logger.info(f"Downloading RP7 {year} from {url}")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    raw = resp.json()

    if isinstance(raw, list):
        data = raw
    elif isinstance(raw, dict):
        data = raw.get("data", raw.get("rows", []))
    else:
        raise ValueError(f"Unexpected API response type: {type(raw)}")

    df = pd.DataFrame(data)
    return df.rename(columns=RP7_COL_MAP)


rp7_sample = load_rp7_year(2022)
print(f"Shape: {rp7_sample.shape}")
print(f"Columns: {rp7_sample.columns.tolist()}")
rp7_sample.head()

In [ ]:
%%time
# Load all available RP7 years (discovered from NGQS metadata above)
rp7_all = pd.concat(
    [load_rp7_year(yr) for yr in RP7_YEARS],
    ignore_index=True,
)
print(f"RP7 total shape: {rp7_all.shape}")
rp7_all.dtypes

In [ ]:

def fmt_years(years):
    """Format a sorted list of integers as compact ranges, e.g. [2005,2006,2008] → '2005-2006, 2008'."""
    if not years:
        return "(none)"
    years = sorted(years)
    ranges = []
    start = end = years[0]
    for y in years[1:]:
        if y == end + 1:
            end = y
        else:
            ranges.append(f"{start}-{end}" if start != end else str(start))
            start = end = y
    ranges.append(f"{start}-{end}" if start != end else str(start))
    return ", ".join(ranges)


# Derive year coverage from the actual loaded data
rp7_years_loaded = set(rp7_all["year"].unique())
rp8_years_loaded = set(rp8_all["year"].unique())

OVERLAP_YEARS = sorted(rp7_years_loaded & rp8_years_loaded)
annual_only_years = sorted(rp7_years_loaded - rp8_years_loaded)
monthly_only_years = sorted(rp8_years_loaded - rp7_years_loaded)

print(f"RP7 years loaded:   {fmt_years(rp7_years_loaded)}")
print(f"RP8 years loaded:   {fmt_years(rp8_years_loaded)}")
print(f"Overlap:            {fmt_years(OVERLAP_YEARS)}")
print(f"Annual-only (RP7):  {fmt_years(annual_only_years)}")
print(f"Monthly-only (RP8): {fmt_years(monthly_only_years)}")


## 4. Respondent Comparison

Are there storage fields/operators present in one dataset but not the other?

We use `report_state` + `gas_field_code` + `reservoir_code` as the natural key
(present in both datasets). `company_name` is used for operator-level counts.

In [ ]:
KEY_COLS = ["report_state", "gas_field_code", "reservoir_code"]

# Get unique reservoir identities in each dataset for the overlap period
rp8_overlap = rp8_all[rp8_all["year"].isin(OVERLAP_YEARS)].copy()
rp7_overlap = rp7_all[rp7_all["year"].isin(OVERLAP_YEARS)].copy()

# Convert key cols to consistent string types for merging
for df in [rp8_overlap, rp7_overlap]:
    for c in KEY_COLS:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()

# Unique entities (across all overlap years — i.e. ever appeared)
rp7_entities = rp7_overlap[KEY_COLS].drop_duplicates()
rp8_entities = rp8_overlap[KEY_COLS].drop_duplicates()

rp7_operators = rp7_overlap[["company_name"]].drop_duplicates()
rp8_operators = rp8_overlap[["company_name"]].drop_duplicates()

print("Unique entities (across all overlap years):")
print(f"  Unique reservoirs in RP7: {len(rp7_entities)}")
print(f"  Unique reservoirs in RP8: {len(rp8_entities)}")
print(f"  Unique operators in RP7:  {len(rp7_operators)}")
print(f"  Unique operators in RP8:  {len(rp8_operators)}")

# Year-indexed counts for trend view
rp8_keys = rp8_overlap[KEY_COLS + ["year"]].drop_duplicates()
rp7_keys = rp7_overlap[KEY_COLS + ["year"]].drop_duplicates()

print(f"\nUnique (reservoir, year) combos:")
print(f"  RP8 (monthly): {len(rp8_keys)}")
print(f"  RP7 (annual):  {len(rp7_keys)}")

In [ ]:
# Merge to find what's in each dataset
merged_keys = pd.merge(
    rp7_keys.assign(in_rp7=True),
    rp8_keys.assign(in_rp8=True),
    on=KEY_COLS + ["year"],
    how="outer",
)
merged_keys["in_rp7"] = merged_keys["in_rp7"].notna()
merged_keys["in_rp8"] = merged_keys["in_rp8"].notna()

print("Record presence summary (by year-reservoir):")
print(merged_keys.groupby(["in_rp7", "in_rp8"]).size().rename("count").to_string())

In [ ]:
# Records in RP7 annual but NOT in RP8 monthly
only_in_rp7 = merged_keys[merged_keys["in_rp7"] & ~merged_keys["in_rp8"]]
print(f"Reservoirs in RP7 annual but not RP8 monthly: {len(only_in_rp7)} records")
if len(only_in_rp7) > 0:
    rp7_names = rp7_overlap[["year"] + KEY_COLS + ["company_name", "field_name", "reservoir_name"]].drop_duplicates()
    display(
        only_in_rp7[KEY_COLS + ["year"]]
        .merge(rp7_names, on=KEY_COLS + ["year"], how="left")
        .head(30)
    )

In [ ]:
# Records in RP8 monthly but NOT in RP7 annual
only_in_rp8 = merged_keys[~merged_keys["in_rp7"] & merged_keys["in_rp8"]]
print(f"Reservoirs in RP8 monthly but not RP7 annual: {len(only_in_rp8)} records")
if len(only_in_rp8) > 0:
    rp8_names = rp8_overlap[["year"] + KEY_COLS + ["company_name", "field_name", "reservoir_name"]].drop_duplicates()
    display(
        only_in_rp8[KEY_COLS + ["year"]]
        .merge(rp8_names, on=KEY_COLS + ["year"], how="left")
        .head(30)
    )

## 5. Value Comparison

For reservoirs appearing in both datasets during the overlap period, we ask:
**what does the RP7 annual value represent?**

Three candidates:
- **December snapshot** — the value reported for month 12
- **Annual average** — mean of the 12 monthly RP8 records
- **Annual sum** — sum of all 12 monthly RP8 records

We aggregate RP8 all three ways, merge with RP7, then focus on the subset of
reservoir-years where the monthly value **actually changed** during the year —
those are the only cases where December ≠ mean ≠ sum, giving us a clean signal.

In [ ]:
NUMERIC_COLS = [
    "working_gas_capacity_mcf",
    "base_gas",
    "total_field_capacity_mcf",
    "maximum_daily_delivery_mcf",
]

# Coerce to numeric in both datasets
for col in NUMERIC_COLS:
    if col in rp7_all.columns:
        rp7_all[col] = pd.to_numeric(rp7_all[col], errors="coerce")
    if col in rp8_all.columns:
        rp8_all[col] = pd.to_numeric(rp8_all[col], errors="coerce")

# Restrict both to overlap period, standardize key cols
rp7_cmp = rp7_all[rp7_all["year"].isin(OVERLAP_YEARS)].copy()
rp8_cmp = rp8_all[rp8_all["year"].isin(OVERLAP_YEARS)].copy()

for df in [rp7_cmp, rp8_cmp]:
    for c in KEY_COLS:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()

In [ ]:
# Aggregate RP8 monthly data three ways for each reservoir-year
rp8_by_year = rp8_cmp.groupby(KEY_COLS + ["year"])[
    [c for c in NUMERIC_COLS if c in rp8_cmp.columns]
].agg(["mean", "sum", "last"]).reset_index()

# Flatten multi-level columns
rp8_by_year.columns = [
    "_".join(c).rstrip("_") if c[1] else c[0] for c in rp8_by_year.columns
]

# Also get December values specifically
rp8_dec = rp8_cmp[rp8_cmp["month"] == 12].copy()

In [ ]:
# Merge RP7 annual with RP8 aggregates
rp7_numeric_cols = [c for c in NUMERIC_COLS if c in rp7_cmp.columns]
rp7_for_merge = rp7_cmp[KEY_COLS + ["year"] + rp7_numeric_cols].copy()

comparison = rp7_for_merge.merge(
    rp8_by_year,
    on=KEY_COLS + ["year"],
    how="inner",
    suffixes=("_rp7", ""),
)

# Also merge December values
rp8_dec_numeric_cols = [c for c in rp7_numeric_cols if c in rp8_dec.columns]
rp8_dec_vals = rp8_dec[KEY_COLS + ["year"] + rp8_dec_numeric_cols].rename(
    columns={c: f"{c}_dec" for c in rp8_dec_numeric_cols}
)
comparison = comparison.merge(rp8_dec_vals, on=KEY_COLS + ["year"], how="left")

In [ ]:
change_mask = {}

# ── Step 1: how often does each column change within a year? ─────────────────
variability_rows = []
for col in NUMERIC_COLS:
    if col not in rp8_cmp.columns:
        continue
    nunique = rp8_cmp.groupby(KEY_COLS + ["year"])[col].nunique()
    changed = nunique > 1
    change_mask[col] = changed
    variability_rows.append({
        "column": col,
        "changed": int(changed.sum()),
        "total": int(len(changed)),
        "pct_changed": changed.mean() * 100,
    })

print("How often do monthly values change within a year (overlap period)?")
display(
    pd.DataFrame(variability_rows)
    .set_index("column")
    .style.format({"pct_changed": "{:.1f}%"})
)

# ── Step 2: for changing records only, which aggregation matches RP7? ─────────
winner_rows = []
for col in NUMERIC_COLS:
    if col not in change_mask:
        continue
    rp7_col = f"{col}_rp7" if f"{col}_rp7" in comparison.columns else col
    if rp7_col not in comparison.columns:
        continue

    changed_pairs = change_mask[col][change_mask[col]].reset_index()[KEY_COLS + ["year"]]
    subset = comparison.merge(changed_pairs, on=KEY_COLS + ["year"])
    if len(subset) == 0:
        continue

    err = {
        agg: (subset[rp7_col] - subset[f"{col}_{agg}"]).abs()
        for agg in ["dec", "mean", "last"]
        if f"{col}_{agg}" in subset.columns
    }
    # dropna before idxmin to avoid FutureWarning on all-NA rows
    err_df = pd.DataFrame(err).dropna(how="all")
    best = err_df.idxmin(axis=1)

    winner_rows.append({
        "column": col,
        "n_changing": len(subset),
        **{f"{agg}_wins_%": (best == agg).mean() * 100 for agg in err},
        **{f"{agg}_median_err": err[agg].median() for agg in err},
    })

print("\nAmong reservoir-years where the value changed: which aggregation best matches RP7?")
display(
    pd.DataFrame(winner_rows)
    .set_index("column")
    .style.format({
        **{f"{agg}_wins_%": "{:.1f}%" for agg in ["dec", "mean", "last"]},
        **{f"{agg}_median_err": "{:,.0f}" for agg in ["dec", "mean", "last"]},
    })
    .highlight_max(subset=["dec_wins_%", "mean_wins_%", "last_wins_%"], axis=1, color="lightgreen")
)

# ── Step 3: spot-check a few rows where dec ≠ mean ───────────────────────────
col = "working_gas_capacity_mcf"
rp7_col = f"{col}_rp7" if f"{col}_rp7" in comparison.columns else col
spot = (
    comparison[
        (comparison[f"{col}_dec"] != comparison[f"{col}_mean"]) & comparison[rp7_col].notna()
    ][KEY_COLS + ["year", rp7_col, f"{col}_mean", f"{col}_dec"]]
    .rename(columns={rp7_col: "rp7", f"{col}_mean": "rp8_mean", f"{col}_dec": "rp8_dec"})
    .head(6)
)
print(f"\nSpot-check ({col}): rows where RP8 Dec ≠ RP8 mean")
display(spot)

## 6. Annual-Only Coverage (2005–2013)

RP7 annual data covers 2005–2013, years that are NOT in PUDL's monthly data.
What does this historical coverage look like?

In [ ]:
annual_only = rp7_all[rp7_all["year"].isin(annual_only_years)].copy()

print(f"RP7-only years (no monthly equivalent): {annual_only_years}")
print(f"Total records: {len(annual_only)}")
print()

annual_only.groupby("year").agg(
    n_reservoirs=("id", "nunique"),       # unique reservoir IDs (not just reservoir_code)
    n_operators=("company_name", "nunique"),
    n_states=("report_state", "nunique"),
).rename_axis("year")

In [ ]:
# Total working gas capacity over time (annual data only)
cap_by_year = (
    rp7_all
    .assign(working_gas_capacity_mcf=pd.to_numeric(rp7_all["working_gas_capacity_mcf"], errors="coerce"))
    .groupby("year")["working_gas_capacity_mcf"]
    .sum()
    .rename("total_working_gas_mcf")
)

# Compare to monthly data (December snapshot) for overlap years
monthly_dec_by_year = (
    rp8_all[rp8_all["month"] == 12]
    .assign(working_gas_capacity_mcf=pd.to_numeric(rp8_all["working_gas_capacity_mcf"], errors="coerce"))
    .groupby("year")["working_gas_capacity_mcf"]
    .sum()
    .rename("total_working_gas_mcf_rp8_dec")
)

combined = pd.concat([cap_by_year, monthly_dec_by_year], axis=1)
combined

In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))

combined["total_working_gas_mcf"].dropna().plot(
    ax=ax, marker="o", label="RP7 Annual"
)
combined["total_working_gas_mcf_rp8_dec"].dropna().plot(
    ax=ax, marker="s", linestyle="--", label="RP8 Monthly (December)"
)

ax.set_title("Total Working Gas Capacity (MCF): RP7 Annual vs RP8 Monthly (Dec)")
ax.set_xlabel("Year")
ax.set_ylabel("Working Gas Capacity (MCF)")
ax.legend()
plt.tight_layout()
plt.show()


## 7. Summary and Recommendations

### Coverage

| Dataset | Years | Notes |
|---------|-------|-------|
| RP7 (Annual) | 2005–2022, 2024 | 2023 missing — reason unknown |
| RP8 (Monthly) | 2014–2025 | PUDL currently ingests this |
| Overlap | 2014–2022, 2024 | Both datasets available |
| **RP7-only** | **2005–2013** | **9 years of history not in monthly data** |

### What does the annual value represent?

All four columns are infrastructure/design parameters that are mostly stable throughout the year.
To distinguish "December snapshot" from "annual average", we restrict to the subset of
reservoir-years where the monthly value actually changed, then check which aggregation
best matches RP7:

| Column | % of years that change | Dec wins | Mean wins | Dec median error |
|--------|----------------------|----------|-----------|-----------------|
| `base_gas` | 17.1% | **96.6%** | 2.8% | 0 |
| `working_gas_capacity_mcf` | 6.9% | **94.6%** | 4.8% | 0 |
| `total_field_capacity_mcf` | 6.0% | **96.1%** | 2.7% | 0 |
| `maximum_daily_delivery_mcf` | 2.1% | **90.6%** | 8.3% | 0 |

**Conclusion: RP7 annual values are December (end-of-year) snapshots.** On records where
monthly values changed, December matches RP7 exactly (median error = 0) in 91–97% of cases,
while the annual mean has a median error of tens to hundreds of thousands of MCF.

### Respondent coverage

- Near-perfect overlap: only 11–12 mismatches out of ~4,100 reservoir-year combinations.
- The 12 records in RP7-only all have `gas_field_code=0`/`reservoir_code=0` — **LNG
  peaking/liquefaction facilities** (JAX LNG, Atlanta Gas, Boston Gas, etc.), a distinct
  survey scope from underground storage. PUDL's RP8 pipeline correctly excludes them.
- The 11 records in RP8-only are legitimate underground storage reservoirs absent from
  particular annual reports — minor gaps with no systematic pattern.

### Pre-2014 historical coverage

RP7 shows steady, meaningful working gas capacity for 2005–2013: ~3.9B MCF in 2005 growing
to ~4.75B MCF by 2013, connecting seamlessly to RP8's 4.79B MCF in 2014. Coverage is
~395–420 reservoirs/year across 30 states.

### Recommendations

1. **Archive RP7 in pudl-archiver** — remove the `"Monthly" in report.description` filter
   in `Eia191Archiver.get_resources()` to include RP7.

2. **Ingest RP7 for 2005–2013** — these 9 years extend PUDL's EIA-191 coverage significantly.
   Treat each record as a December end-of-year snapshot (represent as `month=12`).

3. **Exclude `gas_field_code=0` records** — consistent with RP8 scope (underground storage
   only, not LNG facilities).

4. **Note missing 2023** — RP7 has no 2023 data; RP8 fills this year.

In [ ]:
# Verify the 2013 → 2014 handoff is seamless
pre2014 = cap_by_year[cap_by_year.index < 2014]
print(f"RP7 working gas capacity range (2005–2013): {pre2014.min()/1e9:.2f}B – {pre2014.max()/1e9:.2f}B MCF")
print(f"2013 RP7 total:  {cap_by_year[2013]/1e9:.3f}B MCF")
print(f"2014 RP8 Dec total: {monthly_dec_by_year[2014]/1e9:.3f}B MCF")